# Keras + TensorFlow Foundations

This notebook explains the core building blocks of deep learning in Keras and TensorFlow.



## 1. Imports

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model, Sequential
import numpy as np

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


## 2. Understanding tensors and shapes

In deep learning, shapes matter more than people think.

A tensor is just a multi-dimensional array.

Common examples:

- `(batch_size, features)` for tabular data
- `(batch_size, height, width, channels)` for images
- `(batch_size, sequence_length, features)` for sequences

If your shapes are wrong, your model is wrong.

In [2]:
tabular = tf.random.uniform((4, 10))          # 4 samples, 10 features
image_batch = tf.random.uniform((4, 32, 32, 3))  # 4 RGB images
sequence_batch = tf.random.uniform((4, 20, 8))   # 4 sequences, 20 time steps, 8 features

print("Tabular shape:", tabular.shape)
print("Image batch shape:", image_batch.shape)
print("Sequence batch shape:", sequence_batch.shape)

Tabular shape: (4, 10)
Image batch shape: (4, 32, 32, 3)
Sequence batch shape: (4, 20, 8)


## 3. A custom Dense-like layer

A standard Dense layer performs:


\[
y = activation(xW + b)
\]

In this section, we implement such a layer ourselves.

This is important because it shows what Keras layers really do internally:

- create weights
- create bias
- define the forward pass

In [ ]:
class CustomDenseLayer(layers.Layer):
    def __init__(self, units=32):
        super().__init__()
        self.units = units

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer="random_normal",
            trainable=True,
            name="kernel"
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer="zeros",
            trainable=True,
            name="bias"
        )

    def call(self, inputs):
        return tf.nn.relu(tf.matmul(inputs, self.w) + self.b)

## 4. Testing the custom layer

We now pass dummy input through the custom layer.

Input shape:

- batch size = 5
- features = 10

Output shape:

- batch size = 5
- units = 16

In [ ]:
x = tf.random.uniform((5, 10))
custom_dense = CustomDenseLayer(units=16)
y = custom_dense(x)

print("Input shape:", x.shape)
print("Output shape:", y.shape)

## 5. Sequential API example

The Sequential API is the simplest Keras model style.

Use it when the model is a straight stack of layers:

input -> layer -> layer -> output

In [ ]:
sequential_model = Sequential([
    layers.Dense(64, activation="relu", input_shape=(20,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

sequential_model.summary()

## 6. Functional API example

The Functional API is more flexible.

Use it when you need:

- multiple inputs
- multiple outputs
- branching
- residual connections
- attention models
- complex architectures

Unlike Sequential, the Functional API makes the computation graph explicit.

In [ ]:
inputs = tf.keras.Input(shape=(20,))
x = layers.Dense(64, activation="relu")(inputs)
x = layers.Dense(32, activation="relu")(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

functional_model = tf.keras.Model(inputs, outputs)
functional_model.summary()

## 7. Convolutional Neural Networks (CNNs)

CNNs are designed for grid-like data such as images.

A `Conv2D` layer:

- scans the image with filters
- learns local patterns
- detects edges, textures, shapes

This is different from Dense layers, which connect everything to everything.
CNNs use local connectivity.

In [ ]:
cnn_model = Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

cnn_model.summary()

## 8. Why MaxPooling is used

`MaxPooling2D` reduces spatial dimensions.

Example:

- input: 32x32
- after 2x2 pooling: 16x16

Why do this?

- reduce computation
- keep strong features
- introduce some translational robustness

Pooling reduces spatial size. It does not increase it.

## 9. RNNs and LSTMs

RNNs are designed for sequential data.

Examples:

- text
- time series
- speech
- sensor signals

An LSTM processes a sequence step by step and maintains internal memory.

That is the key difference from Transformers:
LSTMs are sequential over time steps.

In [ ]:
rnn_model = Sequential([
    layers.Input(shape=(50, 10)),   # 50 time steps, 10 features
    layers.LSTM(64),
    layers.Dense(1)
])

rnn_model.summary()

## 10. Testing an LSTM with dummy sequence data

We now create fake sequence input and pass it through the model.

Input shape:

- batch size = 8
- sequence length = 50
- features = 10

In [ ]:
x_seq = tf.random.uniform((8, 50, 10))
y_seq = rnn_model(x_seq)

print("Sequence input shape:", x_seq.shape)
print("Sequence output shape:", y_seq.shape)

## 11. Multi-Head Self-Attention

Self-attention allows each token in a sequence to look at all other tokens.

This is the core idea behind Transformers.

The process is:

1. create Query, Key, Value
2. split embeddings into multiple heads
3. compute attention for each head
4. combine the heads again

Multiple heads let the model learn different relationship patterns.

In [ ]:
class MultiHeadSelfAttention(layers.Layer):
    def __init__(self, embed_dim, num_heads=8):
        super().__init__()
        if embed_dim % num_heads != 0:
            raise ValueError("embed_dim must be divisible by num_heads")

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.projection_dim = embed_dim // num_heads

        self.query_dense = layers.Dense(embed_dim)
        self.key_dense = layers.Dense(embed_dim)
        self.value_dense = layers.Dense(embed_dim)
        self.combine_heads = layers.Dense(embed_dim)

    def attention(self, query, key, value):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32)
        scaled_score = score / tf.math.sqrt(dim_key)
        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value)
        return output, weights

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]

        query = self.query_dense(inputs)
        key = self.key_dense(inputs)
        value = self.value_dense(inputs)

        query = self.split_heads(query, batch_size)
        key = self.split_heads(key, batch_size)
        value = self.split_heads(value, batch_size)

        attention_output, attention_weights = self.attention(query, key, value)

        attention_output = tf.transpose(attention_output, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention_output, (batch_size, -1, self.embed_dim))

        output = self.combine_heads(concat_attention)
        return output

## 12. Testing Multi-Head Self-Attention

We feed a sequence of token embeddings through the attention layer.

Input shape:

- batch size = 2
- sequence length = 12
- embed dim = 64

Output shape remains:

- batch size = 2
- sequence length = 12
- embed dim = 64

Attention changes the representation, not the shape.

In [ ]:
attention_layer = MultiHeadSelfAttention(embed_dim=64, num_heads=8)
dummy_tokens = tf.random.uniform((2, 12, 64))
attention_output = attention_layer(dummy_tokens)

print("Attention input shape:", dummy_tokens.shape)
print("Attention output shape:", attention_output.shape)

## 13. Transformer block

A Transformer block usually contains:

1. Multi-head self-attention
2. residual connection
3. layer normalization
4. feed-forward network
5. second residual connection
6. second normalization

This structure is repeated many times in Transformer encoders.

In [ ]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)

        return self.layernorm2(out1 + ffn_output)

## 14. Testing the Transformer block

Again, the block keeps sequence length and embedding size the same.

What changes is the internal representation of each token.

In [ ]:
transformer_block = TransformerBlock(embed_dim=64, num_heads=8, ff_dim=128)
x = tf.random.uniform((2, 12, 64))
y = transformer_block(x, training=False)

print("Transformer block input shape:", x.shape)
print("Transformer block output shape:", y.shape)

## 15. Stacking Transformer blocks into an encoder

A Transformer encoder is just a stack of Transformer blocks.

Each block refines the sequence representation further.

In [ ]:
class TransformerEncoder(layers.Layer):
    def __init__(self, num_layers, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.enc_layers = [
            TransformerBlock(embed_dim, num_heads, ff_dim, rate)
            for _ in range(num_layers)
        ]

    def call(self, inputs, training=False):
        x = inputs
        for layer in self.enc_layers:
            x = layer(x, training=training)
        return x

## 16. Testing the Transformer encoder

We use 4 encoder layers.

The output still has the same shape as the input, but it is now much richer contextually.

In [ ]:
encoder = TransformerEncoder(num_layers=4, embed_dim=64, num_heads=8, ff_dim=128)
x = tf.random.uniform((2, 12, 64))
y = encoder(x, training=False)

print("Encoder input shape:", x.shape)
print("Encoder output shape:", y.shape)